# 2 · Measuring applause

Each keynote's audio is decoded to 32 kHz mono and passed through PANNs
`Cnn14_DecisionLevelMax`, a CNN trained on AudioSet's 527 sound classes. The model emits a
probability per class every 0.32 s; we keep 13 classes (speech, applause, clapping,
cheering, laughter, music, ...) averaged to one row per second. An ovation is a stretch where
the clap score `max(P(applause), P(clapping))` crosses an onset threshold, with hysteresis.

The thresholds are tuned **only** on human-written caption tags (`[Applause]`,
`[Cheering and applause]`, ...) that professional captioners added to official uploads.
Half of the caption-labelled videos tune the thresholds, the other half measure accuracy.

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display
from unpacked_applause.config import FIGURES, PROCESSED, TABLES
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

In [ ]:
validation = json.loads((TABLES / "detector_validation.json").read_text())
{k: v for k, v in validation.items() if not k.endswith("_ids")}

In [ ]:
grid = pd.read_csv(TABLES / "detector_grid.csv")
grid.head(10)

In [ ]:
per_event = pd.read_csv(TABLES / "detector_per_event.csv")
per_event.describe()[["auc", "ap", "label_share"]]

In [ ]:
display(Image(filename=str(FIGURES / "detector_validation.png")))

## An example timeline

Detected ovations (shaded) against the human caption tags (ticks under the axis).

In [ ]:
for p in sorted(FIGURES.glob("timeline_*.png")):
    display(Image(filename=str(p)))

## Event-level applause measures

In [ ]:
features = pd.read_parquet(PROCESSED / "applause_features.parquet")
from unpacked_applause.catalog import load_events
f = load_events()[["event_id", "company", "audience"]].merge(features, on="event_id")
f.groupby("audience")[["applause_share", "applause_per_min", "applause_max_s", "cheer_share"]].median()

In [ ]:
display(Image(filename=str(FIGURES / "applause_by_company.png")))